# Hebrew Acronym Disambiguation — Test-Set Evaluation (Colab)

Runs every arm against the frozen, natural-only `data/splits/test_items.csv`
(395 items, type-disjoint from train/dev) — the one-shot held-out read.

**This is a one-time evaluation.** Do not re-run this notebook after changing
training data or config based on what it shows — that would turn test into a
second dev set. See `NOTES_FOR_SHAKED.md` for the full reasoning.

Arms: random / most_frequent / most_mined / oracle baselines, untrained
DictaBERT (zero-shot [CLS] similarity), dictabertX (fine-tuned, checkpoint
pulled from a GitHub release), qwen2.5:7b (generate + select, via Ollama
installed in this Colab session), gemini (generate + select, needs an API key).

**Runtime:** `Runtime > Change runtime type > T4 GPU` recommended (dictabertX
and untrained DictaBERT run much faster; qwen/gemini are API/CPU-bound either way).

## 1. Clone the repo and install the package

In [ ]:
!git clone https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git
%cd hebrew-acronym-disambiguation
!git log -1 --oneline

In [ ]:
!pip install -e . -q

## 2. Download the test checkpoint

Pulled from a GitHub Release asset, not a git-tracked file — the checkpoint is
368MB (fp16), over GitHub's 100MB file-push limit and `*.pt` is gitignored.
This is the checkpoint already committed to as final (see
`NOTES_FOR_SHAKED.md`): trained on the released 2829-row `train.csv`, seed 43,
1 epoch, cls pooling — the highest-observed-dev-accuracy run (0.661 on the
62-item natural dev), chosen before this test run, not after.

In [ ]:
!mkdir -p weights
!wget -q -O weights/test_checkpoint.pt https://github.com/BenCarmel123/hebrew-acronym-disambiguation/releases/download/test-checkpoint-20261003/dictabert-crossenc-study_v1-newtrain-seed43-20261003.pt
!ls -la weights/

## 3. Set up Gemini API key

Add `GEMINI_API_KEY` as a Colab secret (key icon in the left sidebar) and
enable it for this notebook, or paste it directly below (not recommended —
avoid committing a key to notebook output).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('loaded GEMINI_API_KEY from Colab secrets')
except Exception as e:
    print('could not load from Colab secrets:', e)
    print('set it manually: os.environ["GEMINI_API_KEY"] = "..."')

## 4. Install Ollama and pull qwen2.5:7b

No local-equivalent service exists in a fresh Colab VM, so this installs and
starts Ollama inside the session itself, then pulls the same model used in all
prior local runs. Takes a few minutes the first time.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # give the server a moment to bind before pulling
print('ollama server started, pid', ollama_proc.pid)

In [ ]:
!ollama pull qwen2.5:7b

## 5. Run every arm against test_items.csv

In [ ]:
TEST_ITEMS = 'data/splits/test_items.csv'
CANDIDATES = 'data/mined/candidate_table.csv'
CHECKPOINT = 'weights/test_checkpoint.pt'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

results = []

In [ ]:
# --- baselines (random / most_frequent / most_mined / oracle) ---
from hebrew_acronyms.models.baselines import evaluate as evaluate_baselines, load_signals
from hebrew_acronyms.models.common.pairs import load_rows

rows = load_rows(TEST_ITEMS)
ranks, mined = load_signals(CANDIDATES)
b = evaluate_baselines(rows, ranks, mined)
for arm, key in [('random', 'random'), ('most_frequent', 'most_frequent'),
                 ('most_mined', 'most_mined'), ('oracle', 'oracle')]:
    results.append({'arm': arm, 'accuracy': b[key], 'invalid_rate': None, 'n_items': b['n_items']})
print(results)

In [ ]:
# --- untrained DictaBERT (zero-shot [CLS] cosine similarity) ---
from hebrew_acronyms.models.dictabert_similarity.model import build_model, MODEL_ID
from hebrew_acronyms.models.dictabert_similarity.eval import evaluate as evaluate_dictabert_zeroshot

tok, encoder = build_model(MODEL_ID)
encoder = encoder.to(device)
encoder.eval()
d = evaluate_dictabert_zeroshot(rows, tok, encoder, device)
results.append({'arm': 'dictabert (untrained)', 'accuracy': d['accuracy'],
                'invalid_rate': None, 'n_items': d['n_items']})
print(d)

### dictabertX (fine-tuned)

This checkpoint was saved by `train_dictabert_colab.ipynb` as a plain
`torch.save(state_dict)` in fp16 — not the strict-sidecar format the new
`src/hebrew_acronyms` `load_finetuned` expects. Reconstructs the same small
CrossEncoder architecture directly, same approach as `notebooks/eval_colab_checkpoint.py`.

In [ ]:
import torch.nn as nn
from hebrew_acronyms.models.common.pairs import ACR_OPEN, ACR_CLOSE, find_span, mark_span
from transformers import AutoModel, AutoTokenizer

MAX_LEN = 256

class CrossEncoder(nn.Module):
    def __init__(self, encoder, hidden, pooling='cls', dropout=0.1):
        super().__init__()
        self.encoder = encoder
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.score = nn.Linear(hidden * 2 if pooling == 'concat' else hidden, 1)

    def forward(self, input_ids, attention_mask, token_type_ids, acr_open_id, acr_close_id):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                         token_type_ids=token_type_ids).last_hidden_state
        pooled = h[:, 0]  # cls pooling, matching the saved checkpoint's config
        return self.score(self.dropout(pooled)).squeeze(-1)

def encode_batch(tok, device, acr_open_id, acr_close_id, context, candidates, max_len=MAX_LEN):
    cls_id, sep_id = tok.cls_token_id, tok.sep_token_id
    rows_ = []
    ctx_ids = tok.encode(context, add_special_tokens=False)
    o, c = ctx_ids.index(acr_open_id), ctx_ids.index(acr_close_id)
    for candidate in candidates:
        cand_ids = tok.encode(candidate, add_special_tokens=False)
        budget = max_len - 3 - len(cand_ids)
        lo, hi = 0, len(ctx_ids)
        while (hi - lo) > budget:
            left_room, right_room = o - lo, hi - (c + 1)
            if left_room >= right_room and left_room > 0:
                lo += 1
            elif right_room > 0:
                hi -= 1
            else:
                break
        trimmed = ctx_ids[lo:hi]
        ids = [cls_id] + trimmed + [sep_id] + cand_ids + [sep_id]
        types = [0] * (len(trimmed) + 2) + [1] * (len(cand_ids) + 1)
        rows_.append((ids, types))
    width = max(len(ids) for ids, _ in rows_)
    pad = tok.pad_token_id or 0
    input_ids, attention_mask, token_type_ids = [], [], []
    for ids, types in rows_:
        n = width - len(ids)
        input_ids.append(ids + [pad] * n)
        attention_mask.append([1] * len(ids) + [0] * n)
        token_type_ids.append(types + [0] * n)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long, device=device),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long, device=device),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long, device=device),
    }

@torch.no_grad()
def evaluate_dictabertx(rows, tok, model, acr_open_id, acr_close_id, device):
    correct = total = 0
    for r in rows:
        cands = [c.strip() for c in r['candidates'].split('|') if c.strip()]
        gold = r['gold_expansion'].strip()
        span = find_span(r['sentence'], r['acronym'])
        if len(cands) < 2 or not gold or span is None:
            continue
        marked = mark_span(r['sentence'], span)
        enc = encode_batch(tok, device, acr_open_id, acr_close_id, marked, cands)
        logits = model(**enc, acr_open_id=acr_open_id, acr_close_id=acr_close_id)
        pred = cands[int(torch.argmax(logits).item())]
        correct += int(pred == gold)
        total += 1
    return {'n_items': total, 'accuracy': round(correct / total, 4) if total else 0.0}

dx_tok = AutoTokenizer.from_pretrained('dicta-il/dictabert')
dx_encoder = AutoModel.from_pretrained('dicta-il/dictabert')
n_added = dx_tok.add_special_tokens({'additional_special_tokens': [ACR_OPEN, ACR_CLOSE]})
if n_added:
    dx_encoder.resize_token_embeddings(len(dx_tok))
acr_open_id, acr_close_id = dx_tok.convert_tokens_to_ids([ACR_OPEN, ACR_CLOSE])

dx_model = CrossEncoder(dx_encoder, hidden=dx_encoder.config.hidden_size, pooling='cls').to(device)
state = torch.load(CHECKPOINT, map_location=device, weights_only=True)
state = {k: v.float() if v.is_floating_point() else v for k, v in state.items()}
dx_model.load_state_dict(state, strict=True)
dx_model.eval()

x = evaluate_dictabertx(rows, dx_tok, dx_model, acr_open_id, acr_close_id, device)
results.append({'arm': 'dictabertX (fine-tuned)', 'accuracy': x['accuracy'],
                'invalid_rate': None, 'n_items': x['n_items']})
print(x)

In [ ]:
# --- qwen2.5:7b (generate + select) via the Ollama server started above ---
from hebrew_acronyms.models.qwen.eval import ollama_generate
from hebrew_acronyms.models.common.eval import evaluate as evaluate_llm

for mode in ('generate', 'select'):
    q = evaluate_llm(rows, generate_fn=ollama_generate, mode=mode)
    results.append({'arm': f'qwen ({mode})', 'accuracy': q['accuracy'],
                    'invalid_rate': q['invalid_rate'], 'n_items': q['n_items']})
    print(mode, q['accuracy'], q['invalid_rate'])

In [ ]:
# --- gemini (generate + select) ---
from hebrew_acronyms.models.gemini.eval import gemini_generate

for mode in ('generate', 'select'):
    g = evaluate_llm(rows, generate_fn=gemini_generate, mode=mode)
    results.append({'arm': f'gemini ({mode})', 'accuracy': g['accuracy'],
                    'invalid_rate': g['invalid_rate'], 'n_items': g['n_items']})
    print(mode, g['accuracy'], g['invalid_rate'])

## 6. Results table

In [ ]:
def to_markdown(results):
    lines = ['| Arm | Accuracy | Invalid rate | Items |', '|---|---|---|---|']
    for r in results:
        acc = f"{r['accuracy']:.3f}" if r['accuracy'] is not None else '—'
        inv = f"{r['invalid_rate']:.3f}" if r['invalid_rate'] is not None else '—'
        n = r['n_items'] if r['n_items'] is not None else '—'
        lines.append(f"| {r['arm']} | {acc} | {inv} | {n} |")
    return '\n'.join(lines)

table = to_markdown(results)
print(table)

with open('test_results.md', 'w', encoding='utf-8') as f:
    f.write(f'# Test-set results ({TEST_ITEMS})\n\n')
    f.write(table + '\n')

from google.colab import files
files.download('test_results.md')